# ML Modeler + ML Reviewer — Step 1 Review

Human-run notebook for the **Step 1: Contracts + State** slice of the ML Modeler + ML Reviewer plan.

Plan: `project_planning/sean_step_artifacts/ML_Modeler_Reviewer_Implementation_Plan.md`  
Checklist: `project_planning/sean_step_artifacts/ML_Modeler_Reviewer_Checklist.md`

Run the cells top to bottom to:
- resolve the repo root even if Jupyter starts in `notebooks/`
- inspect the 5 new modeler-decision contracts and the `MLReviewOutput` phase field
- round-trip each contract against a representative payload
- confirm `PipelineState` gained the new modeling keys with no collisions
- run the focused regression tests that must still pass

This notebook is intentionally small — it only covers Step 1 scope. Later slices (Step 2 modeler modes, Step 3 reviewer modes, Step 4 prompts+router) get their own review notebooks.

In [1]:
from pathlib import Path
import json
import subprocess
from pprint import pprint

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the repo root from the current notebook working directory.")

from multi_agent_ds.core.contracts import (
    BaselineDecision,
    TuningDecision,
    LearningRateDecision,
    FeatureSelectionDecision,
    ModelingVerdict,
    MLReviewOutput,
    ModelingDecisionReview,
)
from multi_agent_ds.orchestration.state import PipelineState

ROOT = resolve_repo_root()
print("Repo root:", ROOT)
print("Notebook cwd:", Path.cwd().resolve())

def run_pytest(args: list[str]) -> None:
    cmd = ["uv", "run", "pytest", *args]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=ROOT)
    if completed.returncode != 0:
        raise RuntimeError(f"pytest failed with exit code {completed.returncode}")


Repo root: /Users/seanlewis/DataspellProjects/multi_agent_ds
Notebook cwd: /Users/seanlewis/DataspellProjects/multi_agent_ds/notebooks


## 1. Inspect the new contract shapes

Each cell below prints the JSON schema for one of the new contracts. Read each schema and confirm it captures the decision the modeler will hand to the reviewer at that phase.

**Human review questions per contract:**
- Does the contract cover every field the reviewer needs to verdict on?
- Is any field missing that the downstream agent will need?
- Are the field names consistent with existing contracts (e.g., `reasoning`, `summary`, `algorithm`)?

In [2]:
print("BaselineDecision — after train_with_defaults, which algorithms are worth tuning?")
pprint(BaselineDecision.model_json_schema())


BaselineDecision — after train_with_defaults, which algorithms are worth tuning?
{'description': "Modeler's decision after reviewing baseline results.",
 'properties': {'algorithms_to_drop': {'items': {'type': 'string'},
                                       'title': 'Algorithms To Drop',
                                       'type': 'array'},
                'algorithms_to_tune': {'items': {'type': 'string'},
                                       'title': 'Algorithms To Tune',
                                       'type': 'array'},
                'reasoning': {'title': 'Reasoning', 'type': 'string'},
                'summary': {'title': 'Summary', 'type': 'string'}},
 'required': ['summary', 'reasoning'],
 'title': 'BaselineDecision',
 'type': 'object'}


In [3]:
print("TuningDecision — after tune_algorithm, do we adopt the Optuna params?")
pprint(TuningDecision.model_json_schema())


TuningDecision — after tune_algorithm, do we adopt the Optuna params?
{'description': "Modeler's decision after reviewing Optuna tuning results.",
 'properties': {'accept_tuned_params': {'title': 'Accept Tuned Params',
                                        'type': 'boolean'},
                'algorithm': {'title': 'Algorithm', 'type': 'string'},
                'chosen_params': {'additionalProperties': True,
                                  'title': 'Chosen Params',
                                  'type': 'object'},
                'reasoning': {'title': 'Reasoning', 'type': 'string'}},
 'required': ['algorithm', 'accept_tuned_params', 'reasoning'],
 'title': 'TuningDecision',
 'type': 'object'}


In [4]:
print("LearningRateDecision — after adjust_learning_rate, do we keep the lowered rate?")
pprint(LearningRateDecision.model_json_schema())


LearningRateDecision — after adjust_learning_rate, do we keep the lowered rate?
{'description': "Modeler's decision after adjusting learning rate.",
 'properties': {'algorithm': {'title': 'Algorithm', 'type': 'string'},
                'chosen_learning_rate': {'title': 'Chosen Learning Rate',
                                         'type': 'number'},
                'chosen_n_estimators': {'title': 'Chosen N Estimators',
                                        'type': 'integer'},
                'keep_adjustment': {'title': 'Keep Adjustment',
                                    'type': 'boolean'},
                'reasoning': {'title': 'Reasoning', 'type': 'string'}},
 'required': ['algorithm',
              'keep_adjustment',
              'chosen_learning_rate',
              'chosen_n_estimators',
              'reasoning'],
 'title': 'LearningRateDecision',
 'type': 'object'}


In [5]:
print("FeatureSelectionDecision — after train_with_feature_subset, do we accept the reduced set?")
pprint(FeatureSelectionDecision.model_json_schema())


FeatureSelectionDecision — after train_with_feature_subset, do we accept the reduced set?
{'description': "Modeler's decision after reviewing permutation importance.",
 'properties': {'accept_subset': {'title': 'Accept Subset', 'type': 'boolean'},
                'algorithm': {'title': 'Algorithm', 'type': 'string'},
                'dropped_features': {'items': {'type': 'string'},
                                     'title': 'Dropped Features',
                                     'type': 'array'},
                'kept_features': {'items': {'type': 'string'},
                                  'title': 'Kept Features',
                                  'type': 'array'},
                'reasoning': {'title': 'Reasoning', 'type': 'string'}},
 'required': ['algorithm', 'accept_subset', 'reasoning'],
 'title': 'FeatureSelectionDecision',
 'type': 'object'}


In [6]:
print("ModelingVerdict — final cross-algorithm recommendation at the end of the loop")
pprint(ModelingVerdict.model_json_schema())


ModelingVerdict — final cross-algorithm recommendation at the end of the loop
{'description': 'Final cross-algorithm recommendation from the ml_modeler.',
 'properties': {'best_algorithm': {'title': 'Best Algorithm', 'type': 'string'},
                'final_metrics': {'additionalProperties': {'additionalProperties': {'type': 'number'},
                                                           'type': 'object'},
                                  'title': 'Final Metrics',
                                  'type': 'object'},
                'justification': {'title': 'Justification', 'type': 'string'},
                'next_action': {'default': 'proceed_to_evaluation',
                                'enum': ['proceed_to_evaluation',
                                         'revise_modeling'],
                                'title': 'Next Action',
                                'type': 'string'},
                'ranked_algorithms': {'items': {'type': 'string'},
                      

In [7]:
print("MLReviewOutput — reused for reviewer verdicts at every phase; new optional phase field")
pprint(MLReviewOutput.model_json_schema())


MLReviewOutput — reused for reviewer verdicts at every phase; new optional phase field
{'$defs': {'ModelingDecisionReview': {'description': 'Review of one modeling '
                                                     'decision.',
                                      'properties': {'classification': {'enum': ['scientific',
                                                                                 'art',
                                                                                 'mixed'],
                                                                        'title': 'Classification',
                                                                        'type': 'string'},
                                                     'decision': {'title': 'Decision',
                                                                  'type': 'string'},
                                                     'mathematical_basis': {'title': 'Mathematical '
                               

## 2. Round-trip each contract

Each cell constructs a representative payload, validates it through the contract, and dumps it back to a dict.  
Use this to confirm the contracts accept realistic modeler / reviewer outputs and reject malformed ones.

**Human review questions:**
- Do these example payloads match what you expect the LLM to produce at each phase?
- Are any default values (`default_factory=list`, `next_action` defaults) surprising?

In [8]:
baseline = BaselineDecision(
    summary="Both algorithms cleared the baseline CV floor; lightgbm leads by ~0.04 gini.",
    algorithms_to_tune=["lightgbm", "logistic_regression"],
    algorithms_to_drop=[],
    reasoning="Both are within tuning range. Keep both to compare at the final verdict.",
)
pprint(baseline.model_dump())


{'algorithms_to_drop': [],
 'algorithms_to_tune': ['lightgbm', 'logistic_regression'],
 'reasoning': 'Both are within tuning range. Keep both to compare at the final '
              'verdict.',
 'summary': 'Both algorithms cleared the baseline CV floor; lightgbm leads by '
            '~0.04 gini.'}


In [9]:
tuning = TuningDecision(
    algorithm="lightgbm",
    accept_tuned_params=True,
    chosen_params={"max_depth": 7, "num_leaves": 63, "reg_alpha": 0.1},
    reasoning="Optuna best score improved CV gini by 0.03 with stable param importances.",
)
pprint(tuning.model_dump())


{'accept_tuned_params': True,
 'algorithm': 'lightgbm',
 'chosen_params': {'max_depth': 7, 'num_leaves': 63, 'reg_alpha': 0.1},
 'reasoning': 'Optuna best score improved CV gini by 0.03 with stable param '
              'importances.'}


In [10]:
lr = LearningRateDecision(
    algorithm="lightgbm",
    keep_adjustment=True,
    chosen_learning_rate=0.025,
    chosen_n_estimators=800,
    reasoning="Halving lr and doubling n_estimators improved test gini without widening the CV/test gap.",
)
pprint(lr.model_dump())


{'algorithm': 'lightgbm',
 'chosen_learning_rate': 0.025,
 'chosen_n_estimators': 800,
 'keep_adjustment': True,
 'reasoning': 'Halving lr and doubling n_estimators improved test gini without '
              'widening the CV/test gap.'}


In [11]:
feature_selection = FeatureSelectionDecision(
    algorithm="lightgbm",
    accept_subset=False,
    kept_features=["age", "income", "credit_score", "region"],
    dropped_features=["postal_code"],
    reasoning="Reduced model matched full-feature gini but test AUC dropped 0.01; not worth the feature loss.",
)
pprint(feature_selection.model_dump())


{'accept_subset': False,
 'algorithm': 'lightgbm',
 'dropped_features': ['postal_code'],
 'kept_features': ['age', 'income', 'credit_score', 'region'],
 'reasoning': 'Reduced model matched full-feature gini but test AUC dropped '
              '0.01; not worth the feature loss.'}


In [12]:
verdict = ModelingVerdict(
    summary="LightGBM outperforms logistic regression on every metric after tuning.",
    best_algorithm="lightgbm",
    ranked_algorithms=["lightgbm", "logistic_regression"],
    final_metrics={
        "lightgbm": {"gini": 0.64, "roc_auc": 0.82, "ase": 0.18},
        "logistic_regression": {"gini": 0.52, "roc_auc": 0.76, "ase": 0.21},
    },
    justification="LightGBM wins on gini by 0.12 and closes the gap vs. Bayes-optimal MSE.",
    next_action="proceed_to_evaluation",
)
pprint(verdict.model_dump())


{'best_algorithm': 'lightgbm',
 'final_metrics': {'lightgbm': {'ase': 0.18, 'gini': 0.64, 'roc_auc': 0.82},
                   'logistic_regression': {'ase': 0.21,
                                           'gini': 0.52,
                                           'roc_auc': 0.76}},
 'justification': 'LightGBM wins on gini by 0.12 and closes the gap vs. '
                  'Bayes-optimal MSE.',
 'next_action': 'proceed_to_evaluation',
 'ranked_algorithms': ['lightgbm', 'logistic_regression'],
 'summary': 'LightGBM outperforms logistic regression on every metric after '
            'tuning.'}


In [13]:
revise_review = MLReviewOutput(
    summary="Tuning accepted too aggressively without a CV/test gap check.",
    approved=False,
    next_action="revise_modeling",
    phase="tuning",
    decisions=[
        ModelingDecisionReview(
            decision="Accept tuned LightGBM params",
            classification="mixed",
            mathematical_basis="Optuna best score alone — no held-out test comparison.",
            reasoning_quality="weak",
            revision_questions=[
                "What does the test-set gini look like at the tuned params?",
                "Is the CV/test gap widening?",
            ],
        )
    ],
)
pprint(revise_review.model_dump())

approve_review = MLReviewOutput(
    summary="Baseline recommendation is sound.",
    approved=True,
    next_action="accept",
    phase="baseline",
)
pprint(approve_review.model_dump())


{'approved': False,
 'decisions': [{'classification': 'mixed',
                'decision': 'Accept tuned LightGBM params',
                'mathematical_basis': 'Optuna best score alone — no held-out '
                                      'test comparison.',
                'reasoning_quality': 'weak',
                'revision_questions': ['What does the test-set gini look like '
                                       'at the tuned params?',
                                       'Is the CV/test gap widening?']}],
 'next_action': 'revise_modeling',
 'phase': 'tuning',
 'summary': 'Tuning accepted too aggressively without a CV/test gap check.'}
{'approved': True,
 'decisions': [],
 'next_action': 'accept',
 'phase': 'baseline',
 'summary': 'Baseline recommendation is sound.'}


## 3. Confirm PipelineState changes

The new state keys are `modeling_verdict` and `should_revise_modeling`. Existing `modeling_results`, `modeling_context`, and `ml_review` are reused.

**Human review questions:**
- Do the new keys avoid collision with existing keys (prep, EDA, evaluation, report)?
- Are the types correct (`dict[str, Any]` for the verdict payload, `bool` for the router flag)?

In [14]:
state_keys = PipelineState.__annotations__
new_keys = ["modeling_verdict", "should_revise_modeling"]
existing_modeling_keys = ["modeling_context", "modeling_results", "ml_review"]

print("New keys added in Step 1:")
for k in new_keys:
    print(f"  {k}: {state_keys[k]}")

print("\nExisting modeling-adjacent keys (reused, not added):")
for k in existing_modeling_keys:
    print(f"  {k}: {state_keys[k]}")

print("\nAll PipelineState keys:")
pprint(sorted(state_keys.keys()))


New keys added in Step 1:
  modeling_verdict: ForwardRef('dict[str, Any]', module='multi_agent_ds.orchestration.state')
  should_revise_modeling: ForwardRef('bool', module='multi_agent_ds.orchestration.state')

Existing modeling-adjacent keys (reused, not added):
  modeling_context: ForwardRef('dict[str, Any]', module='multi_agent_ds.orchestration.state')
  modeling_results: ForwardRef('dict[str, Any]', module='multi_agent_ds.orchestration.state')
  ml_review: ForwardRef('dict[str, Any]', module='multi_agent_ds.orchestration.state')

All PipelineState keys:
['agent_decisions',
 'business_review',
 'current_phase',
 'data',
 'data_path',
 'eda_insights',
 'evaluation_result',
 'experiment_report',
 'iteration',
 'ml_review',
 'modeling_context',
 'modeling_results',
 'modeling_verdict',
 'prep_approved',
 'prep_feedback',
 'prep_iteration',
 'prep_plan',
 'prep_result',
 'processed_data_path',
 'processed_eda_approved',
 'processed_eda_business_review',
 'processed_eda_insights',
 'proc

## 4. Regression: focused existing tests still pass

These are the tests that exercise the contracts and agents this step touches indirectly. They must all pass.

In [15]:
run_pytest([
    "tests/test_pre_modeling_review_agents.py",
    "tests/test_cleaning.py",
    "tests/test_feature_engineering.py",
    "tests/test_preparation_workflow.py",
])


Running: uv run pytest tests/test_pre_modeling_review_agents.py tests/test_cleaning.py tests/test_feature_engineering.py tests/test_preparation_workflow.py
============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/seanlewis/DataspellProjects/multi_agent_ds
configfile: pyproject.toml
plugins: cov-7.1.0, langsmith-0.7.32, Faker-40.13.0, anyio-4.13.0
collected 14 items

tests/test_pre_modeling_review_agents.py ....                            [ 28%]
tests/test_cleaning.py ....                                              [ 57%]
tests/test_feature_engineering.py ...                                    [ 78%]
tests/test_preparation_workflow.py ...                                   [100%]

============================== 14 passed in 3.84s ==============================


## 5. Review sign-off

If you are satisfied:
- tick the two Human review checkpoint boxes for Step 1 in `ML_Modeler_Reviewer_Checklist.md`
- then we commit Step 1 and move to Step 2 (ML Modeler agent modeling modes)

If any contract field looks wrong, note it here and we'll revise before committing.